## Import Libraries

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Libraries ready!")

Libraries ready!


## Load Data

In [4]:
df = pd.read_csv('../data/hr_attrition.csv')

print("Dataset loaded!")
print("Rows and Columns:", df.shape)
df.head()

Dataset loaded!
Rows and Columns: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


## Check Data Quality

In [7]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole      

In [6]:
print("\nDuplicate Rows:", df.duplicated().sum())


Duplicate Rows: 0


In [8]:
print("\nData Types:")
print(df.dtypes)


Data Types:
Age                          int64
Attrition                   object
BusinessTravel              object
DailyRate                    int64
Department                  object
DistanceFromHome             int64
Education                    int64
EducationField              object
EmployeeCount                int64
EmployeeNumber               int64
EnvironmentSatisfaction      int64
Gender                      object
HourlyRate                   int64
JobInvolvement               int64
JobLevel                     int64
JobRole                     object
JobSatisfaction              int64
MaritalStatus               object
MonthlyIncome                int64
MonthlyRate                  int64
NumCompaniesWorked           int64
Over18                      object
OverTime                    object
PercentSalaryHike            int64
PerformanceRating            int64
RelationshipSatisfaction     int64
StandardHours                int64
StockOptionLevel             int64
TotalWo

## Drop Useless Columns

In [9]:
# These columns have same value for all rows
# so they are useless for analysis
df.drop(columns=['EmployeeCount',
                 'Over18',
                 'StandardHours'], inplace=True)

print("Useless columns dropped!")
print("New Shape:", df.shape)

Useless columns dropped!
New Shape: (1470, 32)


## Feature Engineering

In [10]:
# Create Tenure Band
# groups employees by how long they have worked
df['Tenure Band'] = pd.cut(df['YearsAtCompany'],
                           bins=[0, 1, 3, 5, 10, 40],
                           labels=['0-1 yr',
                                   '1-3 yrs',
                                   '3-5 yrs',
                                   '5-10 yrs',
                                   '10+ yrs'])

# Create Age Band
df['Age Band'] = pd.cut(df['Age'],
                        bins=[18, 25, 35, 45, 60],
                        labels=['18-25',
                                '26-35',
                                '36-45',
                                '46-60'])

# Salary gap — is employee earning below department average?
dept_avg = df.groupby('Department')['MonthlyIncome'].transform('mean')
df['Salary vs Dept Avg'] = df['MonthlyIncome'].apply(
    lambda x: 'Below Average' if x < dept_avg.mean() 
    else 'Above Average')

# Attrition Risk Score
# combines key factors that cause attrition
df['Attrition Risk Score'] = (
    # overtime adds risk
    (df['OverTime'] == 'Yes').astype(int) * 3 +
    # low job satisfaction adds risk
    (df['JobSatisfaction'] <= 2).astype(int) * 2 +
    # short tenure adds risk
    (df['YearsAtCompany'] <= 1).astype(int) * 2 +
    # below avg salary adds risk
    (df['Salary vs Dept Avg'] == 'Below Average').astype(int) * 2
)

print("New features added!")
df.head()

New features added!


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,...,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Tenure Band,Age Band,Salary vs Dept Avg,Attrition Risk Score
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,...,0,1,6,4,0,5,5-10 yrs,36-45,Below Average,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,...,3,3,10,7,1,7,5-10 yrs,46-60,Below Average,4
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,...,3,3,0,0,0,0,NaN,36-45,Below Average,7
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,...,3,3,8,7,3,0,5-10 yrs,26-35,Below Average,5
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,...,3,3,2,2,2,2,1-3 yrs,26-35,Below Average,4


## Business Summary

In [11]:
total        = len(df)
attrition    = df[df['Attrition'] == 'Yes'].shape[0]
attr_rate    = round(attrition / total * 100, 2)
avg_income   = round(df['MonthlyIncome'].mean(), 2)
avg_tenure   = round(df['YearsAtCompany'].mean(), 2)
overtime_pct = round((df['OverTime'] == 'Yes').sum() / total * 100, 2)

print("=" * 45)
print("HR BUSINESS SUMMARY")
print("=" * 45)
print(f"Total Employees    : {total}")
print(f"Attrition Count    : {attrition}")
print(f"Attrition Rate     : {attr_rate}%")
print(f"Avg Monthly Income : ${avg_income}")
print(f"Avg Tenure         : {avg_tenure} years")
print(f"Overtime %         : {overtime_pct}%")

HR BUSINESS SUMMARY
Total Employees    : 1470
Attrition Count    : 237
Attrition Rate     : 16.12%
Avg Monthly Income : $6502.93
Avg Tenure         : 7.01 years
Overtime %         : 28.3%


## Save Cleaned Data

In [13]:
df.to_csv('../data/hr_cleaned.csv', index=False)
print("Cleaned file saved!")
print("Final Shape:", df.shape)

Cleaned file saved!
Final Shape: (1470, 36)
